# Neural Network From Scratch

**What you'll build:** A 2-layer MLP with manual forward pass, cross-entropy loss, and backpropagation.

**Prerequisites:** NumPy matrix ops, calculus chain rule, supervised learning basics.

## Concept Recap

**Forward pass:** $z_1 = XW_1 + b_1$, $h = \text{ReLU}(z_1)$, $z_2 = hW_2 + b_2$, $\hat{p} = \text{softmax}(z_2)$

**Loss:** cross-entropy $\mathcal{L} = -\frac{1}{n}\sum_{i,c} y_{ic} \log \hat{p}_{ic}$

**Backprop (chain rule):**
- $dz_2 = \hat{p} - y$ (softmax + cross-entropy combined gradient)
- $dW_2 = h^T \cdot dz_2 / n$
- $dh = dz_2 \cdot W_2^T$
- $dz_1 = dh \odot \mathbf{1}[z_1 > 0]$ (ReLU derivative)
- $dW_1 = X^T \cdot dz_1 / n$

**He initialization:** $W \sim \mathcal{N}(0, \sqrt{2/d_{in}})$ — prevents vanishing/exploding gradients with ReLU.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

def relu(x): return np.maximum(0, x)
def softmax(x): e = np.exp(x - x.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)

class MLP:
    def __init__(self, d_in, d_h, d_out, lr=0.01):
        self.W1 = np.random.randn(d_in, d_h) * np.sqrt(2 / d_in)  # He init
        self.b1 = np.zeros(d_h)
        self.W2 = np.random.randn(d_h, d_out) * np.sqrt(2 / d_h)
        self.b2 = np.zeros(d_out)
        self.lr = lr

    def forward(self, X):
        self.X = X
        self.z1 = X @ self.W1 + self.b1
        self.h = relu(self.z1)
        self.z2 = self.h @ self.W2 + self.b2
        return softmax(self.z2)

    def backward(self, y_one_hot):
        n = len(y_one_hot)
        dz2 = self.forward(self.X) - y_one_hot  # softmax + CE gradient
        self.W2 -= self.lr * (self.h.T @ dz2 / n)
        self.b2 -= self.lr * dz2.mean(0)
        dh = dz2 @ self.W2.T
        dz1 = dh * (self.z1 > 0)  # ReLU derivative
        self.W1 -= self.lr * (self.X.T @ dz1 / n)
        self.b1 -= self.lr * dz1.mean(0)

X, y = make_classification(n_samples=1000, n_features=20, n_classes=3,
                            n_informative=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

np.random.seed(42)
net = MLP(20, 64, 3, lr=0.05)
y_oh = np.eye(3)[y_tr]
losses = []
for i in range(500):
    p = net.forward(X_tr)
    loss = -np.mean(np.log(p[np.arange(len(y_tr)), y_tr] + 1e-9))
    losses.append(loss)
    net.backward(y_oh)

preds = net.forward(X_te).argmax(1)
print(f"Test accuracy: {np.mean(preds == y_te):.4f}")

In [ ]:
from sklearn.neural_network import MLPClassifier

clf = MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42).fit(X_tr, y_tr)
print(f"sklearn MLP accuracy: {clf.score(X_te, y_te):.4f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(losses)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Loss')

# 2D PCA projection of test data colored by prediction
X_2d = PCA(n_components=2).fit_transform(X_te)
scatter = ax2.scatter(X_2d[:, 0], X_2d[:, 1], c=preds, cmap='tab10', alpha=0.6)
ax2.set_title('Test Predictions (PCA projection)')
plt.colorbar(scatter, ax=ax2)
plt.tight_layout(); plt.show()

## Exercises

1. **L2 regularization:** Add $\lambda(\|W_1\|^2 + \|W_2\|^2)$ to the loss. Derive and implement the gradient update.
2. **Third layer:** Extend MLP to 3 layers (add W3, b3). Verify backprop still works by comparing with sklearn.
3. **Momentum:** Add a velocity term $v \leftarrow \beta v + \nabla W$ and update $W \leftarrow W - \eta v$. Compare convergence speed.

## Summary

- Forward pass: linear → nonlinear → linear → softmax
- Backprop: chain rule applied layer by layer from output to input
- Softmax + cross-entropy gradient simplifies to $\hat{p} - y$ — the residual

**Next:** [Backpropagation Deep Dive](backpropagation.ipynb)